# ZKSF quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/quickstart.ipynb)

An approximate simulator normally hands back an outcome distribution with no statement
of how far it might be from the truth, even though the error quantities were available
inside the simulation and were simply discarded on the way out.

ZKSF attaches that statement to every result and publishes it as a certificate anyone
can check.

**Part 1 costs nothing and needs no account.** It reads real, already-completed runs
straight from the public API: exact simulation, approximate simulation with a measured
bound, and a run on a real trapped-ion quantum computer. Nothing is simulated for you
here, these all actually happened.

**Part 2 is optional** and runs new jobs against your own account and balance.

- Docs: https://zksf.org/docs
- Protocol paper: https://doi.org/10.5281/zenodo.21851381


---
# Part 1: read real certified runs

Every finished job on ZKSF can be minted into a certificate with a stable id. The record
is public, carries no account information, and is readable three ways: an HTML page for a
person, a PDF to attach to a paper, and JSON for a program. We will use the JSON.

No authentication, no install beyond what Colab already has.


In [ ]:
import requests

API = "https://api.zksf.org"


def certificate(cert_id):
    """Fetch a public certificate record. No auth, no cost."""
    r = requests.get(f"{API}/certify/{cert_id}/json", timeout=30)
    r.raise_for_status()
    return r.json()


def show(cert_id, title):
    c = certificate(cert_id)
    print(f"=== {title} ===")
    print(f"  protocol   : {c['protocol']}  ({c['certificate_type']})")
    print(f"  engine     : {c['engine']}")
    print(f"  method     : {c['method']}")
    print(f"  shots      : {c['shots']}")
    for key in (
        "exact", "truncation_weight", "truncated_coefficient_mass",
        "error_bound", "certified", "converged", "expectation",
        "device", "fidelity", "fidelity_mode",
    ):
        if key in c and c[key] is not None:
            print(f"  {key:<11}: {c[key]}")
    print(f"  verify     : {c['verify_url']}")
    print()
    return c


## 1a. An exact run

A GHZ state entangles three qubits so they always agree: every shot should read `000` or
`111` and nothing in between. Every gate here is Clifford, so the router sent it to the
Stim stabilizer engine, which simulates such circuits **exactly** at any scale.

When a result is exact there is no approximation error to report. The only scatter is
shot noise, the ordinary statistical wobble of finite sampling.


In [ ]:
ghz = show("d904e28af0b441e0", "3-qubit GHZ, exact stabilizer simulation")


## 1b. An approximate run, with a measured bound

Past roughly 30 qubits exact statevector simulation stops, because state size grows as
`2^n`. Everything beyond is approximate, and this is where an error statement starts to
matter.

This run asked the tensor-network engine for a **certified** bound. It ran with state
renormalization disabled, so the final state's norm deficit equals the total weight
discarded by every truncation during the simulation. That quantity is read directly off
the result rather than estimated, then converted into a bound on any single outcome
probability.

Note how small the discarded weight is, and that the bound is reported rather than
asserted.


In [ ]:
mps = show("fe529e21d7404ff3", "12-qubit GHZ, tensor network, certified bound")


## 1c. Well past the exact limit

Tensor networks are not the only route. Pauli propagation works in the Heisenberg
picture, evolving the *observable* instead of the state and truncating small
coefficients. For the right question it reaches hundreds of qubits.

This is a **192-qubit** GHZ state, measuring the all-Z expectation value. The exact answer
is known analytically to be exactly `+1`, so you can check the result by hand. No Pauli
terms were discarded, so the reported bound is zero.


In [ ]:
pauli = show("5b8b2c4309d44d41", "192-qubit GHZ, Pauli propagation")


## 1d. A real quantum computer

The runs above are classical simulations. This one is not: it ran on **IonQ Forte-1**, a
trapped-ion quantum processor.

Hardware needs a different question. Not "how good is the approximation", but "how close
did the physical device get to the ideal answer". That is the **ZHF-v0.1** protocol. For
circuits small enough to also simulate exactly, it reports the measured Hellinger
fidelity between what the device actually produced and the exact ideal distribution,
where 1.0 would be identical.

The counts are raw. No error mitigation, no post-selection, no readout correction. The
gap from 1.0 is the real physical noise of a real machine.


In [ ]:
qpu = show("df1d4c698a954051", "2-qubit Bell state, IonQ Forte-1 hardware")


## 1e. Side by side

Four runs, four different accuracy regimes, each one independently checkable by anyone
who has the link.


In [ ]:
for c in (ghz, mps, pauli, qpu):
    if c["protocol"].startswith("ZHF"):
        claim = f"hardware fidelity {c.get('fidelity')}"
    elif c.get("exact"):
        claim = "exact, shot noise only"
    else:
        claim = f"error bound {c.get('error_bound')}"
    print(f"{c['engine']:<16} {c['protocol']:<10} {claim}")


### What the fields mean

| Field | Meaning |
|---|---|
| `protocol` | `ZCC-v0.1` for simulation accuracy, `ZHF-v0.1` for hardware fidelity |
| `exact` | The engine made no approximation at all |
| `truncation_weight` | Total probability weight discarded during simulation |
| `error_bound` | How far a single outcome probability may be from the truth |
| `converged` | Whether doubling the resource budget changed the answer |
| `fidelity` | Hardware only: measured overlap with the exact ideal distribution |
| `circuit_sha256` | Identifies the circuit without revealing it |

Every certificate also carries a plain-language section explaining how that specific
result was produced. Open any `verify_url` printed above to read it.

**On the limits.** The certified bound is an empirical result, not a theorem. It held
across 334 runs at sizes where the exact answer is computable, including 290 built
specifically to break it, and the closest approach reached 49 percent of its value. Above
20 qubits it cannot be checked by direct comparison and no such evidence exists. Where a
hard ceiling is required, use an exact or stabilizer engine. The full argument, including
the case where the derivation does not extend, is in
[the paper](https://doi.org/10.5281/zenodo.21851381).


---
# Part 2: run your own

Everything above was free. From here on, jobs run against your account and spend from
your balance.

The circuits below are small and cost a fraction of a cent each, and every one is priced
with a free `estimate()` call first. The hardware cell at the end is left switched off,
because that one is not a fraction of a cent.


In [ ]:
!pip install -q qsim-sdk


Create an account at [app.zksf.org](https://app.zksf.org) and copy your API token.

`getpass` keeps it out of the notebook's saved output, which matters here: notebooks get
shared and forked with their outputs intact. Never paste a token into a cell directly.


In [ ]:
import getpass

import qsim_sdk

client = qsim_sdk.Client(token=getpass.getpass("Paste your ZKSF API token: "))


## 2a. Estimate before you spend

`estimate()` is free and instant. It reports which engine would be chosen, roughly how
long, roughly what cost, and why. It is also the only way to discover that a circuit
would be **rejected** without submitting it.


In [ ]:
from qiskit import QuantumCircuit


def ghz_circuit(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure_all()
    return qc


est = client.estimate(ghz_circuit(40), shots=1000)
print("engine  :", est["engine"])
print("seconds :", est["predicted_seconds"])
print("cost    : $", est["predicted_cost_usd"])
print("reason  :", est["reason"])


## 2b. Run it

40 qubits is well past the exact limit, but a GHZ state is highly structured, so a
tensor-network method handles it comfortably.


In [ ]:
job = client.run(ghz_circuit(40), shots=1000)

print("counts    :", job["result"]["counts"])
print("error_info:", job["result"]["error_info"])


## 2c. Ask for a measured bound

By default an approximate run is checked by convergence: the circuit is simulated again at
double the resource budget and the shift in outcome probabilities is reported. That is
evidence, not a bound.

`certified=True` asks for the stronger statement demonstrated in part 1b. It costs more,
which is why it is opt-in per job.


In [ ]:
job = client.run(ghz_circuit(12), shots=1000, engine="mps.quimb.cpu", certified=True)
print(job["result"]["error_info"])


## 2d. Mint your own certificate

The same call the console's download button makes. The resulting record is public and
readable by anyone, exactly like the four in part 1.


In [ ]:
resp = requests.post(
    f"{API}/jobs/{job['id']}/certificate",
    headers={"Authorization": client._http.headers["Authorization"]},
    timeout=60,
)
resp.raise_for_status()
cert = resp.json()

print("certificate:", cert["cert_id"], cert["protocol"])
print("verify     :", cert["verify_url"])
print("pdf        :", cert["verify_url"] + "/pdf")
print("json       :", cert["verify_url"] + "/json")


## 2e. Real hardware

**Left switched off deliberately.** Hardware is billed per task and per shot at provider
cost, it is orders of magnitude more expensive than anything above, and queue times are
measured in hours rather than seconds. Run `estimate()` first, look at the number, and
only then uncomment.

Available: `qpu.rigetti` (Cepheus-1, 108 qubits, superconducting) and `qpu.ionq`
(Forte-1, trapped ion, the machine that produced the certificate in part 1d).


In [ ]:
# est = client.estimate(ghz_circuit(3), shots=50, engine="qpu.rigetti")
# print("cost: $", est["predicted_cost_usd"])

# job = client.run(ghz_circuit(3), shots=50, engine="qpu.rigetti")
# print(job["result"]["counts"])


---
## Where to go next

- **[Docs](https://zksf.org/docs)** for the full engine list, pricing, and when each engine
  is the right choice.
- **[CERTIFICATION.md](https://github.com/official-dvl/zksf/blob/main/docs/CERTIFICATION.md)**
  for what a bound does and does not assert. The limitations section is the honest part.
- **[The paper](https://doi.org/10.5281/zenodo.21851381)** for the derivations and the
  adversarial testing.
- **Android app**: the same service from a phone,
  [on Google Play](https://play.google.com/store/apps/details?id=com.quantumcomputing.app).

Questions, bugs, or a circuit you think should work but does not: info@zksf.org
